# V26A forward motion-prediction ranking ablation

Frozen, inference-free intervention-sensitivity audit on exactly 16 opened V24.3 regressions.

**Question:** Does replacing forward motion-prediction-error ranking with physical-step ranking recover a meaningful fraction of the 436 attributed losses without unbounded collateral associations?

Candidate generation, reverse mutuality, greedy uniqueness, pruning, routing, edge-confidence inputs, and official metric logic remain frozen. This notebook authorizes no tuning, selector, production mutation, submission, or deployment.

## 1. Configure Repository Paths

Pin the source revision and discover the validated V25 archive plus the competition labels. No detector inference is rerun.

In [ ]:
from pathlib import Path
import gzip
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

EXPECTED_COMMIT = "dd2598dc3f5fb1dc7352f844749b307195b13c12"
ACCEPTED_V25_ARCHIVES = {
    (16_331_934, "e0d765d513890261c0214f12071e236a3191d29d306c58b91b537304459a006c"),
    (16_306_805, "0984e11446817f83b678612c5f79a4ed9c9a7fb94aa150f250e28149e9199d21"),
}
SAMPLE_IDS = [
    "6bba_05b6850b", "6bba_23af9eeb", "6bba_2540cd90", "6bba_2646afc7",
    "6bba_372c8cb8", "6bba_3c5691b6", "6bba_5b28472a", "6bba_5f89039d",
    "6bba_718b21f9", "6bba_76db78c1", "6bba_96833384", "6bba_b204cac7",
    "6bba_d0fc38b5", "6bba_d5eae175", "6bba_ed9377fd", "6bba_fc516dc6",
]

ROOT = Path("/tmp/Atabey")
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(
    ["git", "clone", "--filter=blob:none", "https://github.com/drosadocastro-bit/Atabey.git", str(ROOT)],
    check=True,
)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", EXPECTED_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == EXPECTED_COMMIT

input_root = Path("/kaggle/input")
archive_matches = sorted(input_root.rglob("v25_upstream_forensics_outputs.zip"))
assert len(archive_matches) == 1, archive_matches
V25_ARCHIVE = archive_matches[0]
V25_ARCHIVE_SHA256 = hashlib.sha256(V25_ARCHIVE.read_bytes()).hexdigest()
archive_identity = (V25_ARCHIVE.stat().st_size, V25_ARCHIVE_SHA256)
assert archive_identity in ACCEPTED_V25_ARCHIVES, archive_identity

first_gt_matches = sorted(input_root.rglob(f"{SAMPLE_IDS[0]}.geff"))
assert len(first_gt_matches) == 1, first_gt_matches
TRAIN_DIR = first_gt_matches[0].parent
missing_gt = [sample_id for sample_id in SAMPLE_IDS if not (TRAIN_DIR / f"{sample_id}.geff").exists()]
assert not missing_gt, missing_gt

OUTPUT_A = Path("/kaggle/working/v26a_run_a")
OUTPUT_B = Path("/kaggle/working/v26a_run_b")
print({"commit": actual_commit, "archive": str(V25_ARCHIVE), "archive_identity": archive_identity, "train_dir": str(TRAIN_DIR)})

## 2. Load V25 Association Data

Load the immutable V25 records, normalize sample identities, and verify the complete cohort before any intervention code runs.

In [ ]:
with zipfile.ZipFile(V25_ARCHIVE) as archive:
    sample_entries = sorted(
        name for name in archive.namelist() if name.startswith("run/samples/") and name.endswith(".json.gz")
    )
    assert len(sample_entries) == 16, sample_entries
    v25_records = {
        Path(name).name.removesuffix(".json.gz"): json.loads(gzip.decompress(archive.read(name)))
        for name in sample_entries
    }

assert sorted(v25_records) == sorted(SAMPLE_IDS)
required_record_fields = {
    "sample_id", "coordinate_count", "graph_signatures", "association_audit",
    "visualization", "official_correspondence", "v19_credited_v24_3_lost_edges",
}
for sample_id, record in v25_records.items():
    assert record["sample_id"] == sample_id
    assert required_record_fields <= set(record)
    assert record["graph_mutated"] is False
    assert record["score_claim"] is False
print({"loaded_samples": len(v25_records), "lost_edges": sum(len(row["v19_credited_v24_3_lost_edges"]) for row in v25_records.values())})

## 3. Validate Data Integrity

Install the exact numerical and official-evaluator runtime, validate source pins, then prove all archived identifiers, graph inputs, and relationship records satisfy the frozen contract.

In [ ]:
pinned_official = [
    "git+https://github.com/royerlab/tracksdata.git@39dccf3a243e44274759468cb31b2ad9e7fc1d09",
    "git+https://github.com/royerlab/kaggle-cell-tracking-competition.git@075fc5f5a52d11077f9dc2b074644618f26939e2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pinned_official], check=True)
runtime_packages = [
    "bidict>=0.23.1", "blosc2", "dask", "geff>=1.1.3.1.1", "ilpy>=0.5.1",
    "imagecodecs", "numba", "numcodecs>=0.13", "numpy==2.2.6", "scipy==1.16.3",
    "polars>=1.36.0", "psygnal>=0.14.0", "pyarrow", "rich", "rustworkx>=0.17.1",
    "scikit-image>=0.24.0", "sqlalchemy>=2", "tqdm", "typing-extensions", "zarr>=3.0.10",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *runtime_packages], check=True)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = os.pathsep.join([str(ROOT / "src"), str(ROOT / "scripts")])
probe = subprocess.run(
    [sys.executable, "-c", "import json, numpy, scipy; print(json.dumps({'numpy': numpy.__version__, 'scipy': scipy.__version__}, sort_keys=True))"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
)
print(probe.stdout.strip())
subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(ROOT / "tests/test_v26_a_forward_ranking_contract.py"),
        str(ROOT / "tests/test_v26_a_forward_ranking_shadow.py"),
        str(ROOT / "tests/test_v26_a_forward_ranking_runner.py"),
        "--tb=short", "-q",
    ],
    check=True,
    cwd=ROOT,
    env=RUN_ENV,
)

## 4. Compare Associations with Baseline

Run the single physical-step forward-ranking substitution over the frozen detections. The runner first reproduces all 48 recorded baseline graph hashes, then compares V26A with V24.3 through the pinned official evaluator.

In [ ]:
for output_dir in (OUTPUT_A, OUTPUT_B):
    if output_dir.exists():
        shutil.rmtree(output_dir)

runner_command = [
    sys.executable,
    str(ROOT / "scripts/run_v26_a_forward_ranking_ablation.py"),
    "--train-dir", str(TRAIN_DIR),
    "--v25-archive", str(V25_ARCHIVE),
    "--contract", str(ROOT / "tests/fixtures/v26_a_forward_ranking_ablation.json"),
]
subprocess.run(
    [*runner_command, "--output-dir", str(OUTPUT_A)],
    check=True,
    cwd=ROOT,
    env=RUN_ENV,
)
summary_a = json.loads((OUTPUT_A / "summary.json").read_text(encoding="utf-8"))
assert summary_a["sample_count"] == 16
assert summary_a["deterministic_replay"] is True
assert summary_a["production_tuning_authorized"] is False
assert summary_a["submission_authorized"] is False
print(json.dumps(summary_a["aggregate_edge_transitions"], indent=2, sort_keys=True))

## 5. Identify Orphaned and Duplicate Records

Fail closed on duplicate edge transitions, malformed endpoint pairs, conflicting added/removed records, or incomplete per-sample output.

In [ ]:
def load_sample_results(output_dir: Path) -> dict[str, dict]:
    results = {}
    for path in sorted((output_dir / "samples").glob("*.json.gz")):
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            row = json.load(handle)
        results[row["sample_id"]] = row
    return results

sample_results_a = load_sample_results(OUTPUT_A)
assert sorted(sample_results_a) == sorted(SAMPLE_IDS)
edge_list_fields = {
    "recovered_v19_credited_edges", "displaced_v24_3_credited_edges",
    "newly_credited_edges", "newly_incorrect_edges", "removed_incorrect_edges",
    "added_prediction_edges", "removed_prediction_edges",
}
for sample_id, result in sample_results_a.items():
    ledger = result["transition_ledger"]
    for field in edge_list_fields:
        edges = ledger[field]
        assert all(len(edge) == 2 and all(endpoint is not None for endpoint in edge) for edge in edges), (sample_id, field)
        assert len(edges) == len({tuple(edge) for edge in edges}), (sample_id, field)
    assert not ({tuple(edge) for edge in ledger["added_prediction_edges"]} & {tuple(edge) for edge in ledger["removed_prediction_edges"]})
print({"validated_sample_ledgers": len(sample_results_a), "duplicate_or_orphan_records": 0})

## 6. Analyze Association Changes

Expose aggregate recovery, collateral, pruning, official metric, and mixed-sample behavior. The binding decision comes from the preregistered multi-condition gate, not score alone.

In [ ]:
import pandas as pd

sample_table = pd.DataFrame(summary_a["sample_results"]).set_index("sample_id")
metric_deltas = pd.json_normalize(sample_table["official_metric_delta"]).set_index(sample_table.index)
transition_counts = pd.json_normalize(sample_table["transition_counts"]).set_index(sample_table.index)
analysis_table = pd.concat([transition_counts, metric_deltas.add_prefix("metric_delta.")], axis=1)
display(analysis_table)
display(pd.DataFrame([summary_a["aggregate_edge_transitions"]]))
print(json.dumps(summary_a["interest_gate"], indent=2, sort_keys=True))

## 7. Export Audit Results

Run an independent second replay, compare all scientific fields after removing operational timing/memory telemetry, and bundle the first run with a provenance record for review. Hardware timing is retained but is not treated as deterministic evidence.

In [ ]:
subprocess.run(
    [*runner_command, "--output-dir", str(OUTPUT_B)],
    check=True,
    cwd=ROOT,
    env=RUN_ENV,
)
sample_results_b = load_sample_results(OUTPUT_B)

def scientific_payload(result: dict) -> dict:
    clean = dict(result)
    clean.pop("runtime_seconds", None)
    clean.pop("peak_python_tracemalloc_bytes", None)
    return clean

for sample_id in SAMPLE_IDS:
    assert scientific_payload(sample_results_a[sample_id]) == scientific_payload(sample_results_b[sample_id]), sample_id

run_record = {
    "status": "V26A_INTERVENTION_SENSITIVITY_COMPLETE",
    "atabey_commit": actual_commit,
    "v25_archive_bytes": archive_identity[0],
    "v25_archive_sha256": archive_identity[1],
    "sample_count": len(SAMPLE_IDS),
    "scientific_replay_deterministic": True,
    "decision": summary_a["interest_gate"]["decision"],
    "production_tuning_authorized": False,
    "submission_authorized": False,
}
(OUTPUT_A / "notebook_run_record.json").write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
archive_base = Path("/kaggle/working/v26a_forward_ranking_ablation_outputs")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_A))
print({**run_record, "output_archive": str(archive_path), "output_sha256": hashlib.sha256(archive_path.read_bytes()).hexdigest()})